# Demostración D3 Álvaro — ejecución real y trazas
Prepara la T4 y ejecuta las celdas de instalación, carga e indexación antes de grabar.
Sube `D3_alvaro_ejecucion.zip` en el Paso 2. La consulta siguiente llama a Qwen3-4B;
el caso H03 posterior es una inspección explícita de una traza histórica.
No se ejecuta el lote de 75 preguntas ni se conecta Drive en este notebook.


# Entregable 3 — Sistema de 4 agentes sobre Qwen3-4B
**IA Generativa (580694) · UdeC · QA sobre normativa de pregrado (RI-FI, RG, Calendario 2026)**

Este notebook corre el **mismo** sistema de agentes que probamos localmente con un modelo simulado, pero ahora con **Qwen3-4B** de verdad. El código de los agentes vive en archivos del repositorio; los *prompts* de cada agente son archivos versionados aparte (carpeta `prompts/`), como exige el enunciado.

**Qué hace, en orden:** (1) revisa la GPU, (2) instala dependencias, (3) sube el código y los datos (el zip), (4) carga Qwen3-4B en 4 bits, (5) activa el recuperador denso E5, (6) prueba rápida mostrando los mensajes **entre** agentes, (7) corre el benchmark (50 congeladas + 25 difíciles) y guarda respuestas y trazas para **corrección manual**.

> El notebook **no** calcula el accuracy solo: guarda las respuestas con una columna `correcto` vacía para que la corrección sea manual, como hemos hecho en todo el proyecto.

## Paso 0 — Revisar la GPU
Usa un entorno con GPU (Entorno de ejecución → Cambiar tipo → T4).

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memoria total (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('⚠️  Sin GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).')

## Paso 1 — Instalar dependencias
Qwen3 necesita `transformers` reciente; `bitsandbytes` permite cargar el modelo en 4 bits (cabe en una T4). No se toca `pandas` (Colab depende de su versión).

In [ ]:
# Solo se actualiza lo necesario; pandas y el resto quedan en la versión que trae Colab.
%pip install -q -U "transformers>=4.51" accelerate bitsandbytes
%pip install -q sentence-transformers
print('Listo. Si Colab lo pide, reinicia el entorno (Runtime → Restart) y vuelve a correr desde el Paso 2.')

## Paso 2 — Subir el código y los datos
Sube `D3_alvaro_ejecucion.zip` con el botón de selección. También se admite el ZIP original o un ZIP
del repositorio. Se vuelve a pedir el archivo al ejecutar esta celda; los resultados en Drive se conservan.


In [ ]:
# Subir D3_alvaro_ejecucion.zip (incluido en la carpeta D3 alvaro).
# También admite el ZIP original o un ZIP del repositorio completo.
import os, sys, zipfile, tempfile, shutil
from pathlib import Path
from google.colab import files

subidos = files.upload()
zips = [Path('/content') / nombre for nombre in subidos if nombre.lower().endswith('.zip')]
assert len(zips) == 1, 'Sube exactamente un ZIP del trabajo.'
with tempfile.TemporaryDirectory(prefix='d3_alvaro_') as temporal:
    destino = Path(temporal).resolve()
    with zipfile.ZipFile(zips[0]) as archivo:
        for item in archivo.infolist():
            ruta = (destino / item.filename).resolve()
            if not ruta.is_relative_to(destino):
                raise ValueError('El ZIP contiene una ruta fuera de su carpeta.')
        archivo.extractall(destino)
    candidatos = [p.parent.parent for p in destino.rglob('agentes/orquestador.py')
                  if (p.parent.parent / 'prompts/01_enrutador.txt').exists()]
    assert len(candidatos) == 1, f'Se esperaba un sistema; encontrados: {len(candidatos)}'
    BASE = '/content/Deliverable3'
    if Path(BASE).exists():
        shutil.rmtree(BASE)
    shutil.copytree(candidatos[0], BASE, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

PROMPTS = os.path.join(BASE, 'prompts')
AGENTES = os.path.join(BASE, 'agentes')
if AGENTES not in sys.path:
    sys.path.insert(0, AGENTES)
# Recargar módulos si ya se había usado este notebook en la sesión.
for nombre in ('orquestador', 'tools'):
    sys.modules.pop(nombre, None)
print('Sistema preparado:', BASE)
print(open(os.path.join(AGENTES, 'orquestador.py'), encoding='utf-8').readline().strip())


## Paso 3 — Cargar Qwen3-4B en 4 bits
Cuantización NF4. El cómputo usa float16 en una T4 y bfloat16 cuando la GPU lo permite.
Cada llamada usa temperatura 0,7, top-p 0,8, top-k 20 y min-p 0, con thinking desactivado.
La semilla depende del agente y del prompt. Cambiar el prompt también cambia la semilla.
La igualdad de salidas no está garantizada entre versiones de bibliotecas o hardware distintos.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, hashlib

MODELO = 'Qwen/Qwen3-4B'
# La T4 no tiene soporte nativo de bfloat16: en ese caso se calcula en float16.
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map='auto')
model.eval()
print('Modelo cargado:', MODELO, '| cómputo en', DTYPE)

# Presupuesto de tokens de salida por agente (el Redactor y el Verificador razonan en <pensar>).
MAX_TOKENS = {'enrutador': 256, 'recuperador': 160, 'redactor': 1200, 'verificador': 700}
SEMILLA = 2026   # misma semilla base que el D2

def generar(agente, prompt, max_new_tokens=None):
    """Llamada real a Qwen3-4B. Firma f(agente, prompt)->str que espera el orquestador."""
    msgs = [{'role':'user','content':prompt}]
    try:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    max_new_tokens = max_new_tokens or MAX_TOKENS.get(agente, 512)
    semilla = SEMILLA + int(hashlib.sha256((agente + prompt).encode('utf-8')).hexdigest()[:8], 16) % 1_000_000
    torch.manual_seed(semilla)
    inputs = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.8, top_k=20, min_p=0.0,
                             pad_token_id=tok.eos_token_id)
    gen = out[0][inputs.input_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print(generar('prueba', 'Responde solo con la palabra: listo'))

## Paso 4 — Recuperador denso E5
Cargamos `multilingual-e5-small` y precomputamos los vectores de los 201 fragmentos (el campo `texto_embedding` ya viene con el prefijo `passage:`). Reemplazamos el respaldo léxico de la herramienta `buscar` por la búsqueda por coseno, igual que en el Entregable 2.

In [ ]:
from tools import Corpus
from sentence_transformers import SentenceTransformer
import numpy as np

corpus = Corpus(AGENTES)
e5 = SentenceTransformer('intfloat/multilingual-e5-small', device='cuda' if torch.cuda.is_available() else 'cpu')
frag = corpus.frag
P = e5.encode([f['texto_embedding'] for f in frag], normalize_embeddings=True,
              batch_size=64, show_progress_bar=True)

def buscar_e5(consulta, k=5):
    q = e5.encode(['query: ' + str(consulta)], normalize_embeddings=True)[0]
    sims = P @ q
    orden = np.argsort(-sims)
    vistos, salida = set(), []
    for i in orden:
        f = frag[int(i)]
        uid = f.get('unidad_id') or f.get('parent_id')
        if uid in vistos:
            continue
        vistos.add(uid)
        u = corpus.por_id.get(uid, {})
        salida.append({'id': uid, 'texto': u.get('texto', f['texto'])})
        if len(salida) >= k:
            break
    return salida

corpus.buscar = buscar_e5   # inyecta el recuperador denso en la herramienta
print('Recuperador E5 listo. Prueba:')
for h in corpus.buscar('nota minima para aprobar una asignatura', 3):
    print(' ', h['id'], '-', h['texto'][:70])

## Consulta real: mensajes visibles durante la ejecución

In [ ]:
from orquestador import Orquestador
from pathlib import Path
import json, hashlib, importlib.metadata
from datetime import datetime, timezone

class DemoVisible(Orquestador):
    def _log(self, agente, entrada, salida, extra=None):
        super()._log(agente, entrada, salida, extra)
        print(f"\n[{len(self.traza)}] {agente.upper()}", flush=True)
        print(json.dumps(salida, ensure_ascii=False) if isinstance(salida, dict) else salida, flush=True)
        if extra and extra.get('observacion'):
            print('Observación:', json.dumps(extra['observacion'], ensure_ascii=False), flush=True)

pregunta_demo = '¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?'
print('CONSULTA REAL:', pregunta_demo, flush=True)
resultado_demo = DemoVisible(corpus, generar).responder(pregunta_demo)
print('\nRESPUESTA FINAL:', resultado_demo['respuesta'])
print('VEREDICTO INTERNO:', resultado_demo['veredicto'])
registro_demo = {'origen':'inferencia_real','utc':datetime.now(timezone.utc).isoformat(),
    'pregunta':pregunta_demo,'resultado':resultado_demo,
    'modelo':MODELO,'revision_modelo':getattr(model.config,'_commit_hash',None),
    'versiones':{p:importlib.metadata.version(p) for p in ('torch','transformers','sentence-transformers')},
    'fuentes_sha256':{str(p.relative_to(Path(BASE))):hashlib.sha256(p.read_bytes()).hexdigest()
        for carpeta in ('agentes','prompts') for p in sorted((Path(BASE)/carpeta).rglob('*'))
        if p.is_file() and p.suffix in ('.py','.txt','.json','.csv')}}
Path('/content/demo_consulta_real.json').write_text(json.dumps(registro_demo,ensure_ascii=False,indent=2))


## Fallo H03: inspección de una traza histórica, no inferencia nueva

In [ ]:
trazas_guardadas = json.loads((Path(BASE)/'resultados_v6/trazas_D3.json').read_text())
fallo = trazas_guardadas['dificil_25:H03']
print('TRAZA HISTÓRICA v6 — H03 — NO ES UNA NUEVA EJECUCIÓN')
for evento in fallo:
    print(f"\n[{evento['paso']}] {evento['agente'].upper()}")
    salida = evento['salida']
    print(json.dumps(salida,ensure_ascii=False) if isinstance(salida,dict) else salida)


## Descargar la traza de la consulta real para conservarla con el video

In [ ]:
from google.colab import files
files.download('/content/demo_consulta_real.json')
